In [1]:
pip install langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install langchain langchain-community langchain-groq sentence-transformers faiss-cpu chromadb pypdf duckduckgo-search

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_folder = Path("uploaded_pdfs")
pdf_files = sorted(pdf_folder.glob("*.pdf"))

if not pdf_files:
    print(f"No PDF files found in: {pdf_folder.resolve()}")
    docs = []
else:
    docs = []
    for pdf_file in pdf_files:
        loader = PyPDFLoader(str(pdf_file))
        file_docs = loader.load()
        docs.extend(file_docs)
        print(f"Loaded {len(file_docs)} pages from {pdf_file.name}")

    print(f"\nTotal PDFs: {len(pdf_files)}")
    print(f"Total pages loaded: {len(docs)}")

c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9 pages from 02 Anatomy of an IoT Thing.pdf
Loaded 50 pages from 1774694594_L03_-_Agile_Software_Development.pdf

Total PDFs: 2
Total pages loaded: 59


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

splits = text_splitter.split_documents(docs)

print("Total chunks:", len(splits))

Total chunks: 61


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
 )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2154.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
from collections import defaultdict
from pathlib import Path
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

source_to_chunks = defaultdict(list)
for chunk in splits:
    source_path = chunk.metadata.get("source", "unknown")
    source_name = Path(source_path).name
    source_to_chunks[source_name].append(chunk)

pdf_retrievers = {}
for source_name, chunks in source_to_chunks.items():
    source_store = FAISS.from_documents(chunks, embeddings)
    pdf_retrievers[source_name] = source_store.as_retriever(search_kwargs={"k": 4})

print(f"Retriever ready for {len(pdf_retrievers)} PDF(s)")
for source_name, chunks in source_to_chunks.items():
    print(f"- {source_name}: {len(chunks)} chunks")

Retriever ready for 2 PDF(s)
- 02 Anatomy of an IoT Thing.pdf: 10 chunks
- 1774694594_L03_-_Agile_Software_Development.pdf: 51 chunks


In [22]:
class SmartDocAgent:
    def __init__(self, pdf_retrievers, llm, max_history=8):
        self.pdf_retrievers = pdf_retrievers
        self.llm = llm
        self.name = "SmartDoc Agent"
        self.max_history = max_history
        self.history = []
        self.question_history = []
        self.important_points = []

    def _format_history(self):
        if not self.history:
            return "None"
        lines = []
        for item in self.history[-self.max_history:]:
            lines.append(f"Q: {item['question']}")
            lines.append(f"A: {item['answer_summary']}")
        return "\n".join(lines)

    def _add_important(self, note):
        cleaned = note.strip()
        if not cleaned:
            return False
        lower_existing = {p.lower() for p in self.important_points}
        if cleaned.lower() in lower_existing:
            return False
        self.important_points.append(cleaned)
        self.important_points = self.important_points[-20:]
        return True

    def _extract_note_from_query(self, query):
        lowered = query.lower().strip()
        prefixes = ("remember:", "note:", "important:")
        for prefix in prefixes:
            if lowered.startswith(prefix):
                return query[len(prefix):].strip()

        trigger_words = ["important", "remember", "note", "key point", "main point"]
        if any(word in lowered for word in trigger_words):
            return query.strip()
        return None

    def _remember_turn(self, question, answer_text):
        summary = answer_text.replace("\n", " ").strip()[:300]
        self.history.append({"question": question, "answer_summary": summary})
        self.history = self.history[-self.max_history:]

    def show_memory(self):
        recent_questions = self.question_history[-10:]
        question_lines = [f"- {q}" for q in recent_questions] if recent_questions else ["- None"]
        important_lines = [f"- {p}" for p in self.important_points] if self.important_points else ["- None"]
        return (
            "📌 Memory Status\n"
            f"Stored previous questions: {len(self.question_history)}\n"
            f"Stored important notes: {len(self.important_points)}\n\n"
            "Recent Questions:\n" + "\n".join(question_lines) + "\n\n"
            "Important Notes:\n" + "\n".join(important_lines)
        )

    def _enforce_format(self, source_name, answer):
        required = ["📄 Document:", "📌 Summary:", "📚 Key Points:", "💡 Example:"]
        if all(marker in answer for marker in required):
            return answer

        lines = [line.strip(" -•\t") for line in answer.splitlines() if line.strip()]
        summary_points = lines[:3] if lines else ["No clear summary available from model output."]
        key_points = lines[3:9] if len(lines) > 3 else ["No additional key points extracted."]
        example_line = lines[-1] if lines else "No clear example generated."

        summary_text = "\n".join([f"• {p}" for p in summary_points])
        key_points_text = "\n".join([f"• {p}" for p in key_points])

        return (
            f"📄 Document: {source_name}\n\n"
            "📌 Summary:\n"
            f"{summary_text}\n\n"
            "📚 Key Points:\n"
            f"{key_points_text}\n\n"
            "💡 Example:\n"
            f"{example_line}"
        )

    def run(self, query):
        query = query.strip()
        if not query:
            return "Please enter a question."

        if query.lower() in {"show memory", "/memory", "memory"}:
            return self.show_memory()

        self.question_history.append(query)
        self.question_history = self.question_history[-30:]

        note = self._extract_note_from_query(query)
        note_added = self._add_important(note) if note else False

        if query.lower().startswith(("remember:", "note:", "important:")):
            return "Noted. I saved this in memory.\n\n" + self.show_memory()

        if not self.pdf_retrievers:
            return "No PDF retrievers found. Please upload/load PDFs first."

        history_text = self._format_history()
        important_text = "\n".join(self.important_points[-10:]) if self.important_points else "None"

        answers = []
        for source_name, source_retriever in self.pdf_retrievers.items():
            source_docs = source_retriever.invoke(query)
            pdf_context = "\n\n".join([
                f"(Page {d.metadata.get('page', '?')}) {d.page_content}"
                for d in source_docs
            ])

            if not pdf_context.strip():
                answers.append(
                    "\n".join([
                        "=" * 94,
                        f"📄 Document: {source_name}",
                        "",
                        "📌 Summary:",
                        "• No relevant content found for this question.",
                        "",
                        "📚 Key Points:",
                        "• Not enough matching text in this PDF.",
                        "",
                        "💡 Example:",
                        "No clear example available from this PDF context.",
                    ])
                )
                continue

            prompt = f"""
You are {self.name}, a helpful AI assistant.

Rules:
- Use ONLY this PDF context.
- Do NOT guess.
- Keep language simple and clear.
- Add short real-life examples.
- Mention page numbers where possible.
- Output EXACTLY this structure:

📄 Document: {source_name}

📌 Summary:
• ...
• ...
• ...

📚 Key Points:
• ...
• ...
• ...

💡 Example:
...

Conversation History (previous Q&A):
{history_text}

Important User Notes to Remember:
{important_text}

PDF Context:
{pdf_context}

Current Question: {query}
"""

            raw_answer = self.llm.invoke(prompt).content.strip()
            answer = self._enforce_format(source_name, raw_answer)
            answers.append("=" * 94 + "\n" + answer)

        final_answer = "\n\n".join(answers) + f"\n\n{'=' * 94}\n\n❓ Question:\n{query}"
        self._remember_turn(query, final_answer)

        if note_added:
            final_answer = "(Saved one new important note from your query.)\n\n" + final_answer

        return final_answer

In [5]:
from langchain_groq import ChatGroq
import os

os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

llm = ChatGroq(model="llama-3.3-70b-versatile")

In [24]:
query = "What is this document about?"

if (
    "agent" not in globals()
    or not hasattr(agent, "history")
    or not hasattr(agent, "important_points")
    or not hasattr(agent, "question_history")
    or not hasattr(agent, "_enforce_format")
):
    agent = SmartDocAgent(pdf_retrievers=pdf_retrievers, llm=llm)
else:
    agent.pdf_retrievers = pdf_retrievers
    agent.llm = llm

response = agent.run(query)
print(response)

📄 Document: 02 Anatomy of an IoT Thing.pdf

📌 Summary:
• The document discusses the basics of IoT Things, their architecture, and components.
• It highlights the differences between IoT Things and traditional computers.
• The document provides an overview of the software and hardware components of IoT Things.

📚 Key Points:
• IoT Things are designed for environmental interaction and resource efficiency.
• They have unique architecture and functionality compared to traditional computers.
• IoT Things consist of hardware and software components, including microcontrollers, sensors, actuators, and communication protocols.

💡 Example:
For instance, a smart watch is an example of an IoT Thing, which includes sensors like heart rate and motion, actuators like vibration motor, and a microcontroller to process data and manage inputs/outputs, as mentioned on page 0.

📄 Document: 1774694594_L03_-_Agile_Software_Development.pdf

📌 Summary:
• The document is about Agile Software Development, focus

In [17]:
context = "\n\n".join([
    f"(Page {d.metadata.get('page', '?')}) {d.page_content}"
    for d in docs
])

In [ ]:
def format_memory(chat_history):
    recent_q = []
    notes = []

    for item in chat_history[-10:]:
        if item.startswith("User:"):
            recent_q.append(item.replace("User:", "").strip())

    return recent_q, notes

In [ ]:
def format_docs(docs):
    text = []

    for d in docs:
        text.append(f"• {d.page_content} (Page {d.metadata.get('page', 'N/A')})")

    return "\n".join(text)